# Thalika — Professional Colab Studio

This notebook runs the Colab-ready Thalika package with VoxCPM2 on an NVIDIA GPU. It automatically generates long text in short, stable chunks, matches chunk loudness, inserts punctuation-aware pauses, and exports one 48 kHz mono 24-bit PCM WAV.

**Voice consent:** use only your own voice or a voice whose owner has explicitly permitted cloning.

Recommended: Colab **T4 or better**, High-RAM when available. Keep the runtime connected while generating.


In [ ]:
# 1) Upload and extract Thalika_Colab_Ready.zip
from google.colab import files
from pathlib import Path
import shutil, zipfile, os

PROJECT = Path('/content/Thalika')
if not (PROJECT / 'package.json').exists():
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise RuntimeError('Upload Thalika_Colab_Ready.zip')
    staging = Path('/content/thalika-upload')
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as zf:
        zf.extractall(staging)
    candidates = [p.parent for p in staging.rglob('package.json') if (p.parent / 'local-server/server.py').exists()]
    if not candidates:
        raise RuntimeError('The ZIP does not contain a complete Thalika project.')
    shutil.rmtree(PROJECT, ignore_errors=True)
    shutil.copytree(candidates[0], PROJECT)

os.chdir(PROJECT)
print('Project:', PROJECT)


In [ ]:
# 2) Confirm GPU and configure persistent model cache (optional Google Drive)
import os, subprocess, sys

GPU = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(GPU.stdout if GPU.returncode == 0 else GPU.stderr)
if GPU.returncode != 0:
    raise RuntimeError('GPU not available. Runtime > Change runtime type > T4 GPU, then reconnect.')

USE_GOOGLE_DRIVE_CACHE = False  # Change to True to avoid downloading the model after a runtime reset.
if USE_GOOGLE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/ThalikaCache/huggingface'
else:
    os.environ['HF_HOME'] = '/content/huggingface'

os.environ.update({
    'VOXCPM_DEVICE': 'cuda',
    'VOXCPM_TIMESTEPS': '24',
    'VOXCPM_LOAD_DENOISER': '0',
    'HF_HUB_DOWNLOAD_TIMEOUT': '60',
    'HF_XET_HIGH_PERFORMANCE': '1',
    'NEXT_TELEMETRY_DISABLED': '1',
})
print('Python:', sys.version)
print('HF cache:', os.environ['HF_HOME'])


In [ ]:
# 3) Install Node.js 22 and the Thalika web app
import shutil, subprocess, os
from pathlib import Path

def node_major():
    if not shutil.which('node'):
        return 0
    result = subprocess.run(['node', '--version'], capture_output=True, text=True)
    try:
        return int(result.stdout.strip().lstrip('v').split('.')[0])
    except Exception:
        return 0

if node_major() < 22:
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -', shell=True, check=True)
    subprocess.run(['apt-get', 'install', '-y', 'nodejs'], check=True)

print(subprocess.check_output(['node', '--version'], text=True).strip())
subprocess.run(['npm', 'ci'], cwd=PROJECT, check=True)

(PROJECT / '.env.local').write_text('''HF_VOXCPM2_URL=http://127.0.0.1:7860
HF_REQUEST_TIMEOUT=120000
HF_INFERENCE_TIMEOUT=900000
VOXCPM_DEVICE=cuda
VOXCPM_TIMESTEPS=24
VOXCPM_LOAD_DENOISER=0
''', encoding='utf-8')
print('Web dependencies installed. Environment is ready.')


In [ ]:
# 4) Start VoxCPM2 and wait until the model is ready
import os, subprocess, time, requests, signal
from pathlib import Path

MODEL_LOG = Path('/content/thalika-voxcpm.log')
model_log_handle = MODEL_LOG.open('w')
model_process = subprocess.Popen(
    ['bash', 'scripts/voxcpm-local.sh'],
    cwd=PROJECT,
    env={**os.environ, 'VOXCPM_DEVICE': 'cuda', 'VOXCPM_TIMESTEPS': '24'},
    stdout=model_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print('Model PID:', model_process.pid)
print('First run downloads VoxCPM2 (~8 GB). This can take several minutes.')

deadline = time.time() + 45 * 60
last_notice = 0
while time.time() < deadline:
    if model_process.poll() is not None:
        print(MODEL_LOG.read_text(errors='replace')[-8000:])
        raise RuntimeError('VoxCPM2 stopped before becoming ready.')
    try:
        response = requests.get('http://127.0.0.1:7860/gradio_api/info', timeout=5)
        if response.ok:
            print('VoxCPM2 ready at http://127.0.0.1:7860')
            break
    except requests.RequestException:
        pass
    if time.time() - last_notice > 30:
        print(MODEL_LOG.read_text(errors='replace')[-700:])
        last_notice = time.time()
    time.sleep(5)
else:
    raise TimeoutError('Model startup exceeded 45 minutes. Check the log cell below.')


In [ ]:
# 5) Start the Thalika app and open it in Colab
import subprocess, time, requests
from pathlib import Path
from google.colab import output

APP_LOG = Path('/content/thalika-app.log')
app_log_handle = APP_LOG.open('w')
app_process = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--hostname', '0.0.0.0'],
    cwd=PROJECT,
    env=os.environ.copy(),
    stdout=app_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:3000/api/health', timeout=3).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(APP_LOG.read_text(errors='replace')[-6000:])
    raise RuntimeError('Thalika web app did not start.')

print('Thalika is ready. Choose Movie recap / Professional and the desired emotion intensity.')
output.serve_kernel_port_as_window(3000)


## Recommended settings

- **Professional narration:** Professional broadcast + Neutral/Warm + 50–70% intensity + 1.0x + 20–28 quality steps.
- **Movie recap:** Movie recap + Tense/Dramatic + 70–85% intensity + 1.0–1.1x + 24–32 quality steps.
- **Emotional story:** Cinematic storyteller + Warm/Sad/Hopeful + 65–85% intensity + 0.9–1.0x + 24–32 quality steps.

A long paragraph can be pasted as-is. Thalika splits it at punctuation or safe text boundaries, uses one consistency seed for every chunk, matches chunk loudness, and returns one merged WAV.


In [ ]:
# 6) Diagnostics — run this only when something fails
from pathlib import Path
for name in ['/content/thalika-voxcpm.log', '/content/thalika-app.log']:
    path = Path(name)
    print('\n' + '=' * 18, path.name, '=' * 18)
    print(path.read_text(errors='replace')[-8000:] if path.exists() else 'No log yet.')


In [ ]:
# 7) Stop both servers
import os, signal
for process_name in ['app_process', 'model_process']:
    proc = globals().get(process_name)
    if proc and proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print('Stopped', process_name)
